Software Name: ZeroShotACD-Supplemental-Material 

SPDX-FileCopyrightText: Copyright (c) 2025 Orange SA

SPDX-License-Identifier: MIT


This software is distributed under the MIT license,
the text of which is available at https://opensource.org/license/MIT/
or see the "LICENSE" file for more details.


Authors: See CONTRIBUTORS.txt

Software description: The supplemental material for the paper From Zero-Shot to Reward-Aware: Evaluating Prompting and Memory in LLM-Based Cyber Defense Agents".

In [ ]:
import os
from itertools import accumulate

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

## Stats functions

In [ ]:
def get_run_reward(log_file):

    res = []

    with open(log_file, 'r') as logs:
        for line in logs:
            if 'Reward at 30 steps:' in line:
                res.append(float(line[20:-1]))
            if 'Reward at 50 steps:' in line:
                res.append(float(line[20:-1]))
            if 'Reward at 100 steps:' in line:
                res.append(float(line[21:-1]))

    return res

In [ ]:
def get_list_reward(log_file):

  list_reward = []

  is_prompt_passed = False

  with open(log_file, 'r') as logs:
    for line in logs:
      if not is_prompt_passed and '='*200 in line:
        is_prompt_passed = True
      if is_prompt_passed and 'Reward:' in line and 'Final prompt' not in line:
        list_reward.append(float(line[8:-1]))

  return list_reward

In [ ]:
def interquartile_mean(x, axis):
    q1 = np.percentile(x, 25, axis=axis, keepdims=True)
    q3 = np.percentile(x, 75, axis=axis, keepdims=True)
    mask = (x >= q1) & (x <= q3)
    return np.sum(x * mask, axis=axis) / np.sum(mask, axis=axis)

In [ ]:
def get_reward_stats_iqm(log_folder):
    # Get reward at timestep 30, 50 and 100 for each run
    tot_r30 = []
    tot_r50 = []
    tot_r100 = []
    for run_logs_file in os.listdir(log_folder):
        r30, r50, r100 = get_run_reward(log_folder + run_logs_file)
        tot_r30.append(r30)
        tot_r50.append(r50)
        tot_r100.append(r100)

    # Convert to np array
    tot_r30 = np.array(tot_r30)
    tot_r50 = np.array(tot_r50)
    tot_r100 = np.array(tot_r100)
    rewards = np.column_stack([tot_r30, tot_r50, tot_r100])

    # Get BCI for interquartile mean
    bcis = stats.bootstrap(
        (rewards,),
        interquartile_mean,
        axis=0,
        confidence_level=0.95,
        method="percentile",
    )

    # Calculate interquartile mean
    iqms = interquartile_mean(rewards, axis=0)

    # Get min-max
    mins = rewards.min(axis=0)
    maxs = rewards.max(axis=0)

    return iqms, bcis, mins, maxs

In [ ]:
def format_stat(iqms, bcis, mins, maxs):
    print(f"Reward:")
    print(f"30: {iqms[0]:.2f} [{bcis.confidence_interval.low[0]:.2f}, {bcis.confidence_interval.high[0]:.2f}], {mins[0]:.2f} -- {maxs[0]:.2f}")
    print(f"50: {iqms[1]:.2f} [{bcis.confidence_interval.low[1]:.2f}, {bcis.confidence_interval.high[1]:.2f}], {mins[1]:.2f} -- {maxs[1]:.2f}")
    print(f"100: {iqms[2]:.2f} [{bcis.confidence_interval.low[2]:.2f}, {bcis.confidence_interval.high[2]:.2f}], {mins[2]:.2f} -- {maxs[2]:.2f}")

In [ ]:
def get_reward_curve_iqm(log_folder):
    rewards = []
    for exp_logs_file in os.listdir(log_folder):
        list_reward = get_list_reward(log_folder + exp_logs_file)
        cumul_reward = list(accumulate(list_reward))
        rewards.append(cumul_reward)

    rewards = np.array(rewards)
    bcis = stats.bootstrap(
        (rewards,),
        interquartile_mean,
        axis=0,
        confidence_level=0.95,
        method="percentile",
    )

    iqms = interquartile_mean(rewards, axis=0)

    return iqms, bcis

In [ ]:
def convert_to_tikzpicture(means, bcis):
    """Use to get data in a format suited for latex tikzpicture"""
    means_str = ""
    for i, mean in enumerate(means):
        means_str += f"({i}, {mean:.2f}) "

    lower_bcis = ""
    for i, l in enumerate(bcis.confidence_interval.low):
        lower_bcis += f"({i}, {l:.2f}) "

    upper_bcis = ""
    for i, u in enumerate(bcis.confidence_interval.high):
        upper_bcis += f"({i}, {u:.2f}) "

    return means_str, lower_bcis, upper_bcis

In [ ]:
def plot_rewar_count(log_folder):
  tot_r30 = []
  tot_r50 = []
  tot_r100 = []
  for run_logs_file in os.listdir(log_folder):
      r30, r50, r100 = get_run_reward(log_folder + run_logs_file)
      tot_r30.append(r30)
      tot_r50.append(r50)
      tot_r100.append(r100)

  fig, axes = plt.subplots(1, 3, figsize=(15, 4))
  s = log_folder + run_logs_file
  result = s.rsplit("-", 1)[0] + ".log"
  fig.suptitle(f"Reward count for {result}", fontsize=16)

  axes[0].hist(tot_r30, bins=20)
  axes[0].set_title("Reward at 30")

  axes[1].hist(tot_r50, bins=20)
  axes[1].set_title("Reward at 50")

  axes[2].hist(tot_r100, bins=20)
  axes[2].set_title("Reward at 100")

  for ax in axes:
      ax.set_xlabel("Reward")
      ax.set_ylabel("Count")

  plt.tight_layout()
  plt.show()
  print(tot_r30)
  print(tot_r50)
  print(tot_r100)

In [ ]:
def get_latence(log_folder):

  latences = []

  for run_logs_file in os.listdir(log_folder):
    with open(log_folder + run_logs_file, 'r') as logs:
        for line in logs:
            if 'Time:' in line:
                latences.append(float(line[6:-3]))

  return print(f"Mean action time: {np.array(latences).mean():.2f} ± {np.array(latences).std():.2f}, {min(latences):.2f} / {max(latences):.2f}")

In [ ]:
def get_nb_invalid_action(log_folder):
    count = 0
    for run_logs_file in os.listdir(log_folder):
        with open(log_folder + run_logs_file, 'r') as logs:
            for line in logs:
                if "Blue Action (Invalid):" in line:
                    print(line)
                    count += 1
    print(f"Number of invalid action accross all runs: {count}")

## Usage

In [ ]:
log_path = 'llm_agents/logs/'
log_folder = log_path + 'llm-vs-rl/GPT-5/'

Get reward IQM and BCI as well as min and max reward at time step 30, 50 and 100.

In [ ]:
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
format_stat(iqms, bcis, mins, maxs)

Get reward curve with lower and upper bci curves.

In [ ]:
means, bcis = get_reward_curve_iqm(log_folder)
convert_to_tikzpicture(means, bcis)

Plot histogram of rewards count.

In [ ]:
plot_rewar_count(log_folder)

Get mean time with std in second for agent response as well as minimal and maximal observed response time.

In [ ]:
get_latence(log_folder)

Get number of invalide action across an entire experiment.

In [ ]:
get_nb_invalid_action(log_folder)